# Model Training
Tunes the best classifier and regressor with Optuna, retrains on train+val combined, evaluates once on the sealed test set, and saves all artifacts.

In [11]:
import pandas as pd
import numpy as np
import json, joblib, os, shutil, warnings
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)

DATA_DIR     = '../Dataset/Processed/'
ARTIFACT_DIR = 'artifacts/'
os.makedirs(ARTIFACT_DIR, exist_ok=True)
RANDOM_STATE = 42
N_TRIALS     = 50
sns.set_theme(style='whitegrid')

## 1. Load Processed Data

In [12]:
X_train_clf = pd.read_csv(f'{DATA_DIR}X_train_clf.csv')
X_val_clf   = pd.read_csv(f'{DATA_DIR}X_val_clf.csv')
X_test_clf  = pd.read_csv(f'{DATA_DIR}X_test_clf.csv')
y_train_clf = pd.read_csv(f'{DATA_DIR}y_train_clf.csv').squeeze()
y_val_clf   = pd.read_csv(f'{DATA_DIR}y_val_clf.csv').squeeze()
y_test_clf  = pd.read_csv(f'{DATA_DIR}y_test_clf.csv').squeeze()

X_train_reg = pd.read_csv(f'{DATA_DIR}X_train_reg.csv')
X_val_reg   = pd.read_csv(f'{DATA_DIR}X_val_reg.csv')
X_test_reg  = pd.read_csv(f'{DATA_DIR}X_test_reg.csv')
y_train_reg = pd.read_csv(f'{DATA_DIR}y_train_reg.csv').squeeze()
y_val_reg   = pd.read_csv(f'{DATA_DIR}y_val_reg.csv').squeeze()
y_test_reg  = pd.read_csv(f'{DATA_DIR}y_test_reg.csv').squeeze()

X_trainval_clf = pd.concat([X_train_clf, X_val_clf], ignore_index=True)
y_trainval_clf = pd.concat([y_train_clf, y_val_clf], ignore_index=True)
X_trainval_reg = pd.concat([X_train_reg, X_val_reg], ignore_index=True)
y_trainval_reg = pd.concat([y_train_reg, y_val_reg], ignore_index=True)

print(f'Train+Val clf: {X_trainval_clf.shape} | Test clf: {X_test_clf.shape}')
print(f'Train+Val reg: {X_trainval_reg.shape} | Test reg: {X_test_reg.shape}')

Train+Val clf: (128419, 7) | Test clf: (13626, 7)
Train+Val reg: (77210, 6) | Test reg: (13626, 6)


## 2. Hyperparameter Tuning — Classification (Optuna)

In [14]:
def clf_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state':      RANDOM_STATE,
        'n_jobs':            -1,
    }
    model = RandomForestClassifier(**params)
    model.fit(X_train_clf, y_train_clf)
    return f1_score(y_val_clf, model.predict(X_val_clf), average='weighted')

study_clf = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_clf.optimize(clf_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_clf_params = {**study_clf.best_params, 'random_state': RANDOM_STATE, 'n_jobs': -1}
print(f'\nBest classifier params: {best_clf_params}')
print(f'Best validation F1: {study_clf.best_value:.4f}')

  0%|          | 0/50 [00:00<?, ?it/s]


Best classifier params: {'n_estimators': 317, 'max_depth': 20, 'min_samples_split': 13, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}
Best validation F1: 0.8771


## 3. Hyperparameter Tuning — Regression (Optuna)

In [15]:
def reg_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state':      RANDOM_STATE,
        'n_jobs':            -1,
    }
    model = RandomForestRegressor(**params)
    model.fit(X_train_reg, y_train_reg)
    return np.sqrt(mean_squared_error(y_val_reg, model.predict(X_val_reg)))

study_reg = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_reg.optimize(reg_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_reg_params = {**study_reg.best_params, 'random_state': RANDOM_STATE, 'n_jobs': -1}
print(f'\nBest regressor params: {best_reg_params}')
print(f'Best validation RMSE: {study_reg.best_value:.4f}')

  0%|          | 0/50 [00:00<?, ?it/s]


Best regressor params: {'n_estimators': 338, 'max_depth': 11, 'min_samples_split': 4, 'min_samples_leaf': 10, 'max_features': 'log2', 'random_state': 42, 'n_jobs': -1}
Best validation RMSE: 1.0255


## 4. Final Training on Train + Validation Combined

In [16]:
final_clf = RandomForestClassifier(**best_clf_params)
final_clf.fit(X_trainval_clf, y_trainval_clf)
print('Final classifier trained.')

final_reg = RandomForestRegressor(**best_reg_params)
final_reg.fit(X_trainval_reg, y_trainval_reg)
print('Final regressor trained.')

Final classifier trained.
Final regressor trained.


## 5. Test Set Evaluation (sealed — evaluated once)

In [17]:
y_pred_clf  = final_clf.predict(X_test_clf)
y_proba_clf = final_clf.predict_proba(X_test_clf)[:, 1]

clf_test_f1  = f1_score(y_test_clf, y_pred_clf, average='weighted')
clf_test_auc = roc_auc_score(y_test_clf, y_proba_clf)

print('=== Classification — Test Set ===')
print(f'F1 (weighted): {clf_test_f1:.4f}')
print(f'AUC-ROC:       {clf_test_auc:.4f}')
print(classification_report(y_test_clf, y_pred_clf, target_names=['Safe','Hazardous']))

=== Classification — Test Set ===
F1 (weighted): 0.8763
AUC-ROC:       0.9300
              precision    recall  f1-score   support

        Safe       0.97      0.86      0.92     12300
   Hazardous       0.38      0.79      0.52      1326

    accuracy                           0.86     13626
   macro avg       0.68      0.83      0.72     13626
weighted avg       0.92      0.86      0.88     13626



In [18]:
y_pred_reg = final_reg.predict(X_test_reg)

reg_test_r2   = r2_score(y_test_reg, y_pred_reg)
reg_test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
reg_test_mae  = mean_absolute_error(y_test_reg, y_pred_reg)
rmse_km       = np.sqrt(mean_squared_error(np.expm1(y_test_reg), np.expm1(y_pred_reg)))

print('=== Regression — Test Set ===')
print(f'R2:         {reg_test_r2:.4f}')
print(f'RMSE (log): {reg_test_rmse:.4f}')
print(f'MAE  (log): {reg_test_mae:.4f}')
print(f'RMSE (km):  {rmse_km:,.0f} km')

=== Regression — Test Set ===
R2:         0.1668
RMSE (log): 1.0427
MAE  (log): 0.7562
RMSE (km):  22,409,891 km


## 6. Save Artifacts

In [19]:
joblib.dump(final_clf, f'{ARTIFACT_DIR}classifier.joblib')
joblib.dump(final_reg, f'{ARTIFACT_DIR}regressor.joblib')
shutil.copy(f'{DATA_DIR}scaler_clf.joblib', f'{ARTIFACT_DIR}scaler_clf.joblib')
shutil.copy(f'{DATA_DIR}scaler_reg.joblib', f'{ARTIFACT_DIR}scaler_reg.joblib')

feature_names = {
    'clf_features': list(X_train_clf.columns),
    'reg_features': list(X_train_reg.columns),
}
with open(f'{ARTIFACT_DIR}feature_names.json', 'w') as f:
    json.dump(feature_names, f, indent=2)

metadata = {
    'trained_at': datetime.now().isoformat(),
    'classifier': {
        'model': 'RandomForestClassifier',
        'hyperparams': best_clf_params,
        'test_f1_weighted': round(clf_test_f1, 4),
        'test_auc_roc':     round(clf_test_auc, 4),
    },
    'regressor': {
        'model': 'RandomForestRegressor',
        'hyperparams': best_reg_params,
        'test_r2':       round(reg_test_r2, 4),
        'test_rmse_log': round(reg_test_rmse, 4),
        'test_mae_log':  round(reg_test_mae, 4),
        'test_rmse_km':  round(rmse_km, 0),
    },
    'features': feature_names,
}
with open(f'{ARTIFACT_DIR}model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Artifacts saved:')
for fname in sorted(os.listdir(ARTIFACT_DIR)):
    print(f'  {ARTIFACT_DIR}{fname}')

Artifacts saved:
  artifacts/classifier.joblib
  artifacts/feature_names.json
  artifacts/model_metadata.json
  artifacts/regressor.joblib
  artifacts/scaler_clf.joblib
  artifacts/scaler_reg.joblib
